In [1]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
 
from sklearn.preprocessing import PowerTransformer

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 374)
y_train:  (139,)


(99, 376)

# Feature Selection: RENT

In [14]:
selected_features = ["shape_Sphericity",
                     "glrlm_HighGrayLevelRunEmphasis_PET_c04",
                     "shape_MajorAxisLength",
                     "LBP_102_PET"]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [15]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Yeo-Johnson Transformation

In [16]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)

# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [17]:
X_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.761164,16.969770,42.073251,0.000000
1,0.697049,15.598394,24.613845,0.000000
2,0.565792,17.334294,48.030294,0.000034
3,0.684364,14.009277,25.589900,0.000000
4,0.503142,21.202180,34.684750,0.000199
...,...,...,...,...
134,0.742102,16.110383,33.069705,0.000000
135,0.722918,21.249575,41.043692,0.000000
136,0.652963,14.873884,36.618802,0.000000
137,0.724255,21.648860,45.870392,0.000000


In [18]:
X_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,1.124852,0.125487,0.072022,-0.784694
1,0.161153,-0.358143,-1.421627,-0.784694
2,-1.401990,0.250624,0.444135,0.236612
3,-0.012817,-0.946686,-1.314128,-0.784694
4,-1.982774,1.502519,-0.468644,1.784568
...,...,...,...,...
134,0.823106,-0.175124,-0.601731,-0.784694
135,0.532655,1.517088,0.002519,-0.784694
136,-0.421265,-0.622476,-0.316986,-0.784694
137,0.552472,1.639161,0.314722,-0.784694


In [19]:
MAASTRO_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.668072,19.034156,50.002093,0.000026
1,0.669961,11.392444,41.753334,0.000167
2,0.624081,14.567421,44.375483,0.000057
3,0.577624,13.477331,46.115989,0.000000
4,0.630933,16.365554,54.394967,0.000000
...,...,...,...,...
94,0.671754,17.492295,34.218615,0.000000
95,0.632189,14.267900,51.046869,0.000000
96,0.645548,12.143752,50.417953,0.000000
97,0.727488,15.300229,44.901412,0.000000


In [20]:
MAASTRO_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,-0.228627,0.816980,0.557388,0.033184
1,-0.204032,-1.995319,0.050605,1.682719
2,-0.770221,-0.736265,0.221607,0.693043
3,-1.281223,-1.151240,0.329735,-0.784694
4,-0.689674,-0.085023,0.794695,-0.784694
...,...,...,...,...
94,-0.180591,0.304443,-0.506427,-0.784694
95,-0.674760,-0.848664,0.615633,-0.784694
96,-0.513250,-1.682811,0.580714,-0.784694
97,0.600661,-0.466149,0.254712,-0.784694


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [21]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:51:03,448] A new study created in memory with name: no-name-9aa7761f-bdac-4252-b3ab-0d271a65f648


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7364341085271318


[I 2024-04-15 13:51:08,541] A new study created in memory with name: no-name-238395e6-6ee1-4fcf-921f-854ce4e14b71


Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:08,526] Trial 0 finished with value: 0.6859016206049102 and parameters: {}. Best is trial 0 with value: 0.6859016206049102.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6859016206049102], datetime_start=datetime.datetime(2024, 4, 15, 13, 51, 3, 520655), datetime_complete=datetime.datetime(2024, 4, 15, 13, 51, 8, 526528), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6859016206049102


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24820982416811707
Fold 2 IBS: 0.19413929368245514
Fold 3 IBS: 0.21817979235636706
Fold 4 IBS: 0.19747087662809215
Fold 5 IBS: 0.20028642283459286
[I 2024-04-15 13:51:08,829] Trial 0 finished with value: 0.21165724193392482 and parameters: {}. Best is trial 0 with value: 0.21165724193392482.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21165724193392482], datetime_start=datetime.datetime(2024, 4, 15, 13, 51, 8, 613313), datetime_complete=datetime.datetime(2024, 4, 15, 13, 51, 8, 829612), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21165724193392482


In [22]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [23]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.686
train_ibs:  0.212


#### Test

In [24]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [25]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.524
IBS score: 0.292


In [26]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [27]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [28]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:51:09,051] A new study created in memory with name: no-name-6d9b7526-b008-468f-b50f-d86eaebb1445


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-15 13:51:09,403] A new study created in memory with name: no-name-9f2fc26d-3d8b-451b-be5e-0025fddb1747


Fold 1 C-index: 0.5956175298804781
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.7262357414448669
Fold 5 C-index: 0.6802575107296137
[I 2024-04-15 13:51:09,389] Trial 0 finished with value: 0.6592742098498834 and parameters: {}. Best is trial 0 with value: 0.6592742098498834.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6592742098498834], datetime_start=datetime.datetime(2024, 4, 15, 13, 51, 9, 135480), datetime_complete=datetime.datetime(2024, 4, 15, 13, 51, 9, 389072), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6592742098498834


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.247247092638399
Fold 2 IBS: 0.2320398751219593
Fold 3 IBS: 0.22898186742616655
Fold 4 IBS: 0.24197475878533212
Fold 5 IBS: 0.22939558471523763
[I 2024-04-15 13:51:09,614] Trial 0 finished with value: 0.2359278357374189 and parameters: {}. Best is trial 0 with value: 0.2359278357374189.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2359278357374189], datetime_start=datetime.datetime(2024, 4, 15, 13, 51, 9, 437856), datetime_complete=datetime.datetime(2024, 4, 15, 13, 51, 9, 614294), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2359278357374189


In [29]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [30]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.659
train_ibs:  0.236


#### Test

In [31]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [32]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.534


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [33]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [34]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:51:09,809] A new study created in memory with name: no-name-dee45ae6-6415-4abe-b979-3f84f2f44f37


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6595744680851063
Fold 4 C-index: 0.7452471482889734


[I 2024-04-15 13:51:10,240] A new study created in memory with name: no-name-da890083-0514-4a0e-8598-0918dd7c9d5a


Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:10,233] Trial 0 finished with value: 0.6850505567751229 and parameters: {}. Best is trial 0 with value: 0.6850505567751229.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6850505567751229], datetime_start=datetime.datetime(2024, 4, 15, 13, 51, 9, 851776), datetime_complete=datetime.datetime(2024, 4, 15, 13, 51, 10, 233742), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6850505567751229


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24713202318432798
Fold 2 IBS: 0.19357514296294234
Fold 3 IBS: 0.21802432702977978
Fold 4 IBS: 0.1974050690265357
Fold 5 IBS: 0.2001300462867355
[I 2024-04-15 13:51:10,640] Trial 0 finished with value: 0.21125332169806427 and parameters: {}. Best is trial 0 with value: 0.21125332169806427.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21125332169806427], datetime_start=datetime.datetime(2024, 4, 15, 13, 51, 10, 267556), datetime_complete=datetime.datetime(2024, 4, 15, 13, 51, 10, 640199), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21125332169806427


In [35]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [36]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.685
train_ibs:  0.211


#### Test 

In [37]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [38]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.523


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.29


In [39]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [40]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:51:10,876] A new study created in memory with name: no-name-6ebf34f3-17e8-45c3-b277-a3eaa53df025


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6595744680851063
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:11,262] Trial 0 finished with value: 0.6850505567751229 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6850505567751229.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:11,685] Trial 1 finished with value: 0.6859016206049102 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6859016206049102.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:12,098] Trial 2 finished with value: 0.6859016206049102 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:20,653] Trial 24 finished with value: 0.6859016206049102 and parameters: {'l1_ratio': 0.21623254971460304}. Best is trial 10 with value: 0.6866984333539141.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:21,299] Trial 25 finished with value: 0.6866984333539141 and parameters: {'l1_ratio': 0.0853699617771228}. Best is trial 10 with value: 0.6866984333539141.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:21,783] Trial 26 finished with value: 0.6859016206049102 and parameters: {'l1_ratio': 0.17284350892750144}. Best is trial 10 with value: 0.6866984333539141.
Fold 1 C-index: 

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:31,101] Trial 48 finished with value: 0.6859016206049102 and parameters: {'l1_ratio': 0.28913086771769636}. Best is trial 10 with value: 0.6866984333539141.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6638297872340425
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 13:51:31,785] Trial 49 finished with value: 0.6859016206049102 and parameters: {'l1_ratio': 0.3881372941956815}. Best is trial 10 with value: 0.6866984333539141.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.6695278969957081
[I 2024-04-15 13:51:31,955] Trial 50 finished with value: 0.6560490188311844 and parameters: {'l1_ratio': 0.0297385714488726

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.24711400402245015
Fold 2 IBS: 0.1936255360028472
Fold 3 IBS: 0.21775302705493618
Fold 4 IBS: 0.19733991812882137
Fold 5 IBS: 0.20014875551269515
[I 2024-04-15 13:51:51,038] Trial 0 finished with value: 0.21119624814434998 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.21119624814434998.
Fold 1 IBS: 0.2471842830612004
Fold 2 IBS: 0.19368615252951107
Fold 3 IBS: 0.21708862067855675
Fold 4 IBS: 0.19718100727416427
Fold 5 IBS: 0.20018061988835872
[I 2024-04-15 13:51:51,516] Trial 1 finished with value: 0.21106413668635823 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.21106413668635823.
Fold 1 IBS: 0.24723347067992624
Fold 2 IBS: 0.19367590819513153
Fold 3 IBS: 0.21686033606483
Fold 4 IBS: 0.19715383356207036
Fold 5 IBS: 0.20019099705145302
[I 2024-04-15 13:51:52,002] Trial 2 finished with value: 0.21102290911068225 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.21102290911068

Fold 3 IBS: 0.21713038174854807
Fold 4 IBS: 0.19719086208313805
Fold 5 IBS: 0.20017435708736836
[I 2024-04-15 13:52:04,795] Trial 25 finished with value: 0.2110800916600984 and parameters: {'l1_ratio': 0.31702325556753097}. Best is trial 23 with value: 0.2109640931446135.
Fold 1 IBS: 0.24710976507231067
Fold 2 IBS: 0.19356657811592767
Fold 3 IBS: 0.21796758095923752
Fold 4 IBS: 0.19739133670018458
Fold 5 IBS: 0.20014120372422037
[I 2024-04-15 13:52:05,154] Trial 26 finished with value: 0.21123529291437615 and parameters: {'l1_ratio': 0.9272212602815642}. Best is trial 23 with value: 0.2109640931446135.
Fold 1 IBS: 0.2472852642692684
Fold 2 IBS: 0.1937243410615861
Fold 3 IBS: 0.21650645308338287
Fold 4 IBS: 0.19704202120529382
Fold 5 IBS: 0.20021538935636174
[I 2024-04-15 13:52:05,547] Trial 27 finished with value: 0.21095469379517856 and parameters: {'l1_ratio': 0.08658554700947207}. Best is trial 27 with value: 0.21095469379517856.
Fold 1 IBS: 0.24735434592823255
Fold 2 IBS: 0.2267950

Fold 1 IBS: 0.24733058201948163
Fold 2 IBS: 0.19374228550558356
Fold 3 IBS: 0.21646981442818783
Fold 4 IBS: 0.19703304865849025
Fold 5 IBS: 0.20021216636339284
[I 2024-04-15 13:52:15,049] Trial 50 finished with value: 0.2109575793950272 and parameters: {'l1_ratio': 0.09245607769603169}. Best is trial 27 with value: 0.21095469379517856.
Fold 1 IBS: 0.24733608628168227
Fold 2 IBS: 0.19374447388546562
Fold 3 IBS: 0.2164842109301202
Fold 4 IBS: 0.19703652520186807
Fold 5 IBS: 0.20021235699329745
[I 2024-04-15 13:52:15,506] Trial 51 finished with value: 0.21096273065848675 and parameters: {'l1_ratio': 0.09322127061868946}. Best is trial 27 with value: 0.21095469379517856.
Fold 1 IBS: 0.24737783516167797
Fold 2 IBS: 0.22828951614611498
Fold 3 IBS: 0.22876373450842225
Fold 4 IBS: 0.23706684221599025
Fold 5 IBS: 0.22610653553141913
[I 2024-04-15 13:52:15,872] Trial 52 finished with value: 0.23352089271272494 and parameters: {'l1_ratio': 0.04573584965321713}. Best is trial 27 with value: 0.2109

Fold 1 IBS: 0.2473003630703882
Fold 2 IBS: 0.19371534452480685
Fold 3 IBS: 0.21673547099250093
Fold 4 IBS: 0.19709653799375557
Fold 5 IBS: 0.20019762802244337
[I 2024-04-15 13:52:24,477] Trial 75 finished with value: 0.211009068920779 and parameters: {'l1_ratio': 0.16519036404979054}. Best is trial 53 with value: 0.21094387775999937.
Fold 1 IBS: 0.2452490121428647
Fold 2 IBS: 0.22956505914115263
Fold 3 IBS: 0.2288281989762864
Fold 4 IBS: 0.2387039274036472
Fold 5 IBS: 0.2272183201218906
[I 2024-04-15 13:52:24,817] Trial 76 finished with value: 0.23391290355716832 and parameters: {'l1_ratio': 0.02872279649413431}. Best is trial 53 with value: 0.21094387775999937.
Fold 1 IBS: 0.24731727540725035
Fold 2 IBS: 0.22745251816434017
Fold 3 IBS: 0.22872829921003118
Fold 4 IBS: 0.2360114074589735
Fold 5 IBS: 0.20022292848895276
[I 2024-04-15 13:52:25,127] Trial 77 finished with value: 0.22794648574590962 and parameters: {'l1_ratio': 0.05788272087021253}. Best is trial 53 with value: 0.2109438777

In [41]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [42]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.687
train_ibs:  0.211


#### Test

In [43]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [44]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.09799455604980911)

test_cindex : 0.523


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.09047684364202117)

test_ibs:  0.29


In [45]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [46]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:52:34,369] A new study created in memory with name: no-name-42e6c84b-8d72-46b7-b68c-1688080d3762


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.6723404255319149
Fold 4 C-index: 0.7281368821292775
Fold 5 C-index: 0.648068669527897
[I 2024-04-15 13:52:40,148] Trial 0 finished with value: 0.6977075894586338 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6977075894586338.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7276595744680852
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:52:45,304] Trial 1 finished with value: 0.7128781680014569 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 4 C-index: 0.7813688212927756
Fold 5 C-index: 0.703862660944206
[I 2024-04-15 13:53:15,419] Trial 15 finished with value: 0.7359974816455777 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 6, 'min_samples_leaf': 16, 'max_depth': 6, 'n_estimators': 135, 'oob_score': True, 'max_samples': 0.9916077098875709, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2050785814281292, 'warm_start': True}. Best is trial 14 with value: 0.7401429490839831.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.779467680608365
Fold 5 C-index: 0.7060085836909872
[I 2024-04-15 13:53:15,662] Trial 16 finished with value: 0.7424649927687818 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 7, 'oob_score': True, 'max_samples': 0.9632036286697128, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19855793167729388, 'warm_start': True}. Best is trial

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.813953488372093
Fold 3 C-index: 0.8468085106382979
Fold 4 C-index: 0.8174904942965779
Fold 5 C-index: 0.7725321888412017
[I 2024-04-15 13:53:30,594] Trial 30 finished with value: 0.7736629125252518 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 17, 'n_estimators': 369, 'oob_score': True, 'max_samples': 0.6833587310905035, 'max_features': None, 'min_weight_fraction_leaf': 0.008532800179007469, 'warm_start': True}. Best is trial 29 with value: 0.7751733627229969.
Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.8425531914893617
Fold 4 C-index: 0.8136882129277566
Fold 5 C-index: 0.7639484978540773
[I 2024-04-15 13:53:32,788] Trial 31 finished with value: 0.7735002862697369 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 379, 'oob_score': True, 'max_samples': 0.6547593902381754,

...

Fold 1 IBS: 0.22612682993385486
Fold 2 IBS: 0.17223298934748948
Fold 3 IBS: 0.20782358499736456
Fold 4 IBS: 0.20181112716562463
Fold 5 IBS: 0.21350833459590543
[I 2024-04-15 13:58:06,586] Trial 32 finished with value: 0.2043005732080478 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 13, 'max_depth': 8, 'n_estimators': 316, 'oob_score': False, 'max_samples': 0.8753375341925101, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.2230974552568295}. Best is trial 1 with value: 0.20384699429436565.
Fold 1 IBS: 0.22177687649658975
Fold 2 IBS: 0.17630377861515312
Fold 3 IBS: 0.2122309287307206
Fold 4 IBS: 0.2017574245411477
Fold 5 IBS: 0.21470545803951144
[I 2024-04-15 13:58:09,301] Trial 33 finished with value: 0.20535489328462447 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 307, 'oob_score': False, 'max_samples': 0.8780403780133842, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.23790435513187447
Fold 2 IBS: 0.16917310562036944
Fold 3 IBS: 0.2032724536335342
Fold 4 IBS: 0.20275263571784216
Fold 5 IBS: 0.21198800508925086
[I 2024-04-15 13:58:32,786] Trial 48 finished with value: 0.20501811103857426 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 14, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 369, 'oob_score': False, 'max_samples': 0.862646612475437, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.18882619674434536}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.21643739431200149
Fold 2 IBS: 0.18002210069568697
Fold 3 IBS: 0.20963559196115233
Fold 4 IBS: 0.20201711337877098
Fold 5 IBS: 0.21711926927169345
[I 2024-04-15 13:58:33,458] Trial 49 finished with value: 0.20504629392386103 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 12, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 68, 'oob_score': False, 'max_samples': 0.712240935894755, 'max_features': 'sqrt', 'min_weight_fraction

Fold 5 IBS: 0.21281753768850425
[I 2024-04-15 13:58:48,997] Trial 63 finished with value: 0.20478734721351177 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 16, 'min_samples_leaf': 12, 'max_depth': 6, 'n_estimators': 145, 'oob_score': False, 'max_samples': 0.5771231216606082, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.05763095148942208}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.21904071817956408
Fold 2 IBS: 0.17322770907939164
Fold 3 IBS: 0.21071337338690455
Fold 4 IBS: 0.20198953222034577
Fold 5 IBS: 0.21531568867352077
[I 2024-04-15 13:58:49,737] Trial 64 finished with value: 0.20405740430794536 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 2, 'n_estimators': 74, 'oob_score': False, 'max_samples': 0.6331344922813995, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1612709153840626}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.2211019166635484
Fold 2 IBS: 0.175

Fold 3 IBS: 0.19348965127184473
Fold 4 IBS: 0.20729301862706373
Fold 5 IBS: 0.21518309439089522
[I 2024-04-15 13:59:02,366] Trial 79 finished with value: 0.2023912644271136 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 6, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.7940170614934956, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19696931079449848}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.2449767955817733
Fold 2 IBS: 0.172095460075217
Fold 3 IBS: 0.21463433607213933
Fold 4 IBS: 0.21510140082957746
Fold 5 IBS: 0.21527266884222518
[I 2024-04-15 13:59:02,784] Trial 80 finished with value: 0.21241613228018644 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 10, 'max_depth': 6, 'n_estimators': 28, 'oob_score': False, 'max_samples': 0.8170871700419534, 'max_features': None, 'min_weight_fraction_leaf': 0.21125680575069572}. Best is trial 40 with value: 0.20227913

Fold 1 IBS: 0.26932513184118945
Fold 2 IBS: 0.1697323509215535
Fold 3 IBS: 0.20905255797983618
Fold 4 IBS: 0.2258672184082545
Fold 5 IBS: 0.2152407835992728
[I 2024-04-15 13:59:12,243] Trial 95 finished with value: 0.21784360855002127 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 11, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.8547756485942715, 'max_features': None, 'min_weight_fraction_leaf': 0.17594983744388534}. Best is trial 86 with value: 0.20204409612264693.
Fold 1 IBS: 0.22065985212578162
Fold 2 IBS: 0.1724043068676393
Fold 3 IBS: 0.212876495767763
Fold 4 IBS: 0.20196603399913246
Fold 5 IBS: 0.21590968295775706
[I 2024-04-15 13:59:13,436] Trial 96 finished with value: 0.2047632743436147 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 7, 'min_samples_leaf': 15, 'max_depth': 13, 'n_estimators': 61, 'oob_score': False, 'max_samples': 0.7614906038154946, 'max_features': 'log2', 'min_weight_fraction_leaf': 

In [47]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [48]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.83
train_ibs:  0.202


#### Test

In [49]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [50]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=18, max_leaf_nodes=11,
                     max_samples=0.9059465751594485, min_samples_leaf=1,
                     min_weight_fraction_leaf=0.029068528038335394,
                     n_estimators=318, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.544


RandomSurvivalForest(max_depth=13, max_features='log2', max_leaf_nodes=11,
                     max_samples=0.7105780352803006, min_samples_leaf=11,
                     min_samples_split=13,
                     min_weight_fraction_leaf=0.2021626029496649,
                     n_estimators=31, random_state=123)

test_ibs:  0.247


In [51]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [52]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:59:18,402] A new study created in memory with name: no-name-b9f4eb6e-664c-49e6-80f0-382afbebbe1e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.7490494296577946
Fold 5 C-index: 0.6909871244635193
[I 2024-04-15 13:59:19,201] Trial 0 finished with value: 0.7329375617815092 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7329375617815092.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:59:21,069] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:59:39,571] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 20, 'n_estimators': 246, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.12362405012814498, 'min_weight_fraction_leaf': 0.18949444225079395}. Best is trial 8 with value: 0.7362139149697521.
Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7872340425531915
Fold 4 C-index: 0.7604562737642585
Fold 5 C-index: 0.6909871244635193
[I 2024-04-15 13:59:40,484] Trial 17 finished with value: 0.7333650628805521 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 9, 'n_estimators': 362, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8640754037508087, 'min_weight_fraction_leaf': 0.08300984976320

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.7566539923954373
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 14:00:00,261] Trial 31 finished with value: 0.7301852441402965 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 9, 'min_samples_leaf': 10, 'max_depth': 11, 'n_estimators': 288, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8780867392507613, 'min_weight_fraction_leaf': 0.08053342835298444}. Best is trial 8 with value: 0.7362139149697521.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.7829787234042553
Fold 4 C-index: 0.7490494296577946
Fold 5 C-index: 0.6909871244635193
[I 2024-04-15 14:00:01,787] Trial 32 finished with value: 0.7270669981840107 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.2369623605519169
Fold 2 IBS: 0.2015664827514855
Fold 3 IBS: 0.2044196732107329
Fold 4 IBS: 0.2152719150958778
Fold 5 IBS: 0.2123358137500061
[I 2024-04-15 14:03:28,387] Trial 30 finished with value: 0.21411124907200385 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.20923338228172916.
Fold 1 IBS: 0.23451695205118067
Fold 2 IBS: 0.19751010562426458
Fold 3 IBS: 0.2027198084019687
Fold 4 IBS: 0.21411412044401185
Fold 5 IBS: 0.2111761169753505
[I 2024-04-15 14:03:32,623] Trial 31 finished with value: 0.21200742069935527 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.923

Fold 1 IBS: 0.23247724033039832
Fold 2 IBS: 0.18748554961121114
Fold 3 IBS: 0.2090170455982855
Fold 4 IBS: 0.207742375469341
Fold 5 IBS: 0.20889347688505747
[I 2024-04-15 14:04:44,234] Trial 45 finished with value: 0.2091231375788587 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 144, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8505483502716148, 'min_weight_fraction_leaf': 0.09794633817349793}. Best is trial 45 with value: 0.2091231375788587.
Fold 1 IBS: 0.2314286160573986
Fold 2 IBS: 0.1860889745854672
Fold 3 IBS: 0.20638267011548084
Fold 4 IBS: 0.20884587453459477
Fold 5 IBS: 0.20805701088484838
[I 2024-04-15 14:04:46,111] Trial 46 finished with value: 0.20816062923555795 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 138, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.77092

Fold 1 IBS: 0.2376464824369693
Fold 2 IBS: 0.21593994734134395
Fold 3 IBS: 0.22177987321764975
Fold 4 IBS: 0.22790863488656224
Fold 5 IBS: 0.2205529164521157
[I 2024-04-15 14:05:27,604] Trial 60 finished with value: 0.22476557086692814 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 74, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.6041879821929975, 'min_weight_fraction_leaf': 0.1056236625824671}. Best is trial 46 with value: 0.20816062923555795.
Fold 1 IBS: 0.23781793031064252
Fold 2 IBS: 0.1862495208476139
Fold 3 IBS: 0.20248456745784213
Fold 4 IBS: 0.21234711020159586
Fold 5 IBS: 0.2076972639565933
[I 2024-04-15 14:05:30,932] Trial 61 finished with value: 0.20931927855485752 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 19, 'n_estimators': 197, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.71011260

Fold 1 IBS: 0.23693321208405524
Fold 2 IBS: 0.19000973925047765
Fold 3 IBS: 0.20243881773088923
Fold 4 IBS: 0.2110667431713562
Fold 5 IBS: 0.20765848231427028
[I 2024-04-15 14:05:52,816] Trial 75 finished with value: 0.20962139891020973 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 18, 'n_estimators': 104, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.49599352625156035, 'min_weight_fraction_leaf': 0.03138093454428739}. Best is trial 71 with value: 0.20716724376288945.
Fold 1 IBS: 0.2375162774090104
Fold 2 IBS: 0.21696007860868566
Fold 3 IBS: 0.2199964189465905
Fold 4 IBS: 0.2298197447784768
Fold 5 IBS: 0.2208032877666601
[I 2024-04-15 14:05:54,020] Trial 76 finished with value: 0.2250191615018847 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 14, 'min_samples_leaf': 1, 'max_depth': 17, 'n_estimators': 81, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.6498539

Fold 1 IBS: 0.23900229487413618
Fold 2 IBS: 0.1829989350900894
Fold 3 IBS: 0.20504008460974205
Fold 4 IBS: 0.21135329415618298
Fold 5 IBS: 0.20658610615550185
[I 2024-04-15 14:06:25,010] Trial 90 finished with value: 0.20899614297713048 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 16, 'n_estimators': 95, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7812194815989179, 'min_weight_fraction_leaf': 0.052583924158953604}. Best is trial 71 with value: 0.20716724376288945.
Fold 1 IBS: 0.23336269917153504
Fold 2 IBS: 0.18793910685243745
Fold 3 IBS: 0.20227191981491185
Fold 4 IBS: 0.2126322523584538
Fold 5 IBS: 0.20891055719221097
[I 2024-04-15 14:06:28,444] Trial 91 finished with value: 0.2090233070779098 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 170, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7

In [53]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [54]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.764
train_ibs:  0.207


#### Test

In [55]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [56]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=6, max_features=None, max_leaf_nodes=13,
                   max_samples=0.9529576562335005, min_samples_split=15,
                   min_weight_fraction_leaf=0.05156899271012762,
                   n_estimators=376, random_state=123, warm_start=True)

C-index score: 0.543


ExtraSurvivalTrees(max_depth=19, max_features=None, max_leaf_nodes=17,
                   max_samples=0.6638490929451707, min_samples_leaf=2,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.052914450220883674,
                   n_estimators=60, oob_score=True, random_state=123)

IBS: 0.256


In [57]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [58]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 14:06:49,326] A new study created in memory with name: no-name-0c40934b-131e-4e3d-8667-ea42258a92d0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:07:17,403] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:07:28,172] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:13:48,980] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7079199845805899.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:15:12,754] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:36:04,423] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.7079199845805899.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 14:36:44,298] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 16:35:56,530] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 53 with value: 0.2348064399929884.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:37:14,476] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.357604

Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:48:20,074] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 64 with value: 0.23475649727610257.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:48:55,524] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.001312025546225645, 'dropout_rate': 0.8046

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 16:58:59,965] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 64 with value: 0.23475649727610257.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:59:32,166] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209832, 'dropout_rate': 0.625165508

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 17:08:12,791] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7762725560789899, 'learning_rate': 0.0030671608519508686, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.27772784044341225, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 2}. Best is trial 85 with value: 0.23301282639832266.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 17:09:18,135] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9240076064991638, 'learning_rate': 0.036372907201929795, 'dropout_rate': 0.166

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 17:18:06,617] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.06808689488183181, 'dropout_rate': 0.25443633448028735, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 0.5386043790221791, 'min_weight_fraction_leaf': 0.03403879613376609, 'max_features': 'auto', 'min_impurity_decrease': 8.69596280226011e-07, 'validation_fraction': 0.9221483446288596, 'min_samples_split': 20, 'max_leaf_nodes': 12, 'min_samples_leaf': 16, 'max_depth': 2}. Best is trial 85 with value: 0.23301282639832266.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23301282639832266], datetime_start=datetime.datetime(2024, 4, 15, 17, 4, 15, 706328), datetime_complete=datetime.datetime(2024, 4, 15, 17, 5, 21, 943873), params={'subsample': 0.9481727373988897, 'learning_rate': 0.020229752590967

In [59]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [60]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.713
train_ibs:  0.233


#### Test

In [61]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [62]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.022737959076050005,
                                 dropout_rate=0.8285090704895912,
                                 learning_rate=0.0513322635401625, max_depth=2,
                                 max_features='sqrt', max_leaf_nodes=10,
                                 min_impurity_decrease=0.002070496973827786,
                                 min_samples_leaf=15, min_samples_split=19,
                                 min_weight_fraction_leaf=0.3101145382804477,
                                 n_estimators=485, random_state=123,
                                 subsample=0.45214538811231725,
                                 validation_fraction=0.9071319756271838)

C-index score: 0.532


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.229


In [63]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [64]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 17:18:22,069] A new study created in memory with name: no-name-31a20e10-2829-42ef-96f6-d75b2b3070d8


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 17:18:23,014] Trial 0 finished with value: 0.6423526693826518 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6423526693826518.
Fold 1 C-index: 0.6653386454183267
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6425855513307985
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 17:18:30,065] Trial 1 finished with value: 0.6456489898043869 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6456489898043869.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6523605150214592
[I 2024-04-15 17:19:27,563] Trial 19 finished with value: 0.6444529500787335 and parameters: {'subsample': 0.3743552574085504, 'dropout_rate': 0.9964748437560931, 'n_estimators': 303, 'learning_rate': 0.08697733590757546}. Best is trial 15 with value: 0.6532531222787401.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 17:19:33,228] Trial 20 finished with value: 0.6443294235502888 and parameters: {'subsample': 0.179916923106687, 'dropout_rate': 0.6101927969129244, 'n_estimators': 422, 'learning_rate': 0.06631423193408584}. Best is trial 15 with value: 0.6532531222787401.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5659574468085107
Fold

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6523605150214592
[I 2024-04-15 17:20:28,713] Trial 38 finished with value: 0.6434311170524222 and parameters: {'subsample': 0.23557094034920928, 'dropout_rate': 0.29839135236038283, 'n_estimators': 221, 'learning_rate': 0.0775915195100935}. Best is trial 37 with value: 0.656117456606762.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.6738197424892703
[I 2024-04-15 17:20:30,168] Trial 39 finished with value: 0.6577474385800097 and parameters: {'subsample': 0.16797447968278584, 'dropout_rate': 0.48271147924203567, 'n_estimators': 174, 'learning_rate': 0.06892741183938003}. Best is trial 39 with value: 0.6577474385800097.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745


...

Fold 3 IBS: 0.27412770407771303
Fold 4 IBS: 0.2250727043053573
Fold 5 IBS: 0.22804546200325004
[I 2024-04-15 17:25:41,198] Trial 19 finished with value: 0.23081617472029087 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 7 with value: 0.22258329722925968.
Fold 1 IBS: 0.22813963029627435
Fold 2 IBS: 0.21261919492535192
Fold 3 IBS: 0.2328032377684259
Fold 4 IBS: 0.23177471930860935
Fold 5 IBS: 0.21241390693232276
[I 2024-04-15 17:25:42,171] Trial 20 finished with value: 0.22355013784619687 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 7 with value: 0.22258329722925968.
Fold 1 IBS: 0.23294882482388735
Fold 2 IBS: 0.2172731528360657
Fold 3 IBS: 0.23037554967791973
Fold 4 IBS: 0.23393962524579212
Fold 5 IBS: 0.21554329357937044
[I 2024-04-15 17:25:42,935] Trial 21 fin

Fold 4 IBS: 0.23024202553473688
Fold 5 IBS: 0.2195522967050084
[I 2024-04-15 17:26:01,345] Trial 38 finished with value: 0.22655527241132453 and parameters: {'subsample': 0.7340144528858092, 'dropout_rate': 0.13272164755980653, 'n_estimators': 83, 'learning_rate': 0.030782657817155126}. Best is trial 31 with value: 0.22099872671568876.
Fold 1 IBS: 0.2332317850992294
Fold 2 IBS: 0.2182381217008508
Fold 3 IBS: 0.2300773630902979
Fold 4 IBS: 0.23503197229669454
Fold 5 IBS: 0.216451746174902
[I 2024-04-15 17:26:01,802] Trial 39 finished with value: 0.22660619767239493 and parameters: {'subsample': 0.7926935273119843, 'dropout_rate': 0.7752839243709247, 'n_estimators': 33, 'learning_rate': 0.01715036845928685}. Best is trial 31 with value: 0.22099872671568876.
Fold 1 IBS: 0.2602657323081961
Fold 2 IBS: 0.2421911304010384
Fold 3 IBS: 0.3269201446440632
Fold 4 IBS: 0.26764332948211106
Fold 5 IBS: 0.27728378008086335
[I 2024-04-15 17:26:02,992] Trial 40 finished with value: 0.27486082338325446

Fold 5 IBS: 0.28393992071489904
[I 2024-04-15 17:26:14,646] Trial 57 finished with value: 0.2764735346125301 and parameters: {'subsample': 0.2888110376335854, 'dropout_rate': 0.3333949349209909, 'n_estimators': 102, 'learning_rate': 0.07586937026396535}. Best is trial 42 with value: 0.21951257463556278.
Fold 1 IBS: 0.2459075656523321
Fold 2 IBS: 0.20109168779305825
Fold 3 IBS: 0.2909620806928277
Fold 4 IBS: 0.21878444537392763
Fold 5 IBS: 0.23150502717743096
[I 2024-04-15 17:26:15,379] Trial 58 finished with value: 0.23765016133791533 and parameters: {'subsample': 0.22421139413625618, 'dropout_rate': 0.1983473170754516, 'n_estimators': 51, 'learning_rate': 0.08704357466884936}. Best is trial 42 with value: 0.21951257463556278.
Fold 1 IBS: 0.4024929155383612
Fold 2 IBS: 0.32055960805224804
Fold 3 IBS: 0.3911151691769955
Fold 4 IBS: 0.2590462193700875
Fold 5 IBS: 0.3349029305354637
[I 2024-04-15 17:26:20,808] Trial 59 finished with value: 0.3416233685346312 and parameters: {'subsample': 

Fold 5 IBS: 0.21153802617953904
[I 2024-04-15 17:26:45,210] Trial 76 finished with value: 0.2217341801045 and parameters: {'subsample': 0.18496931802695063, 'dropout_rate': 0.31169806189512894, 'n_estimators': 26, 'learning_rate': 0.09087507715614321}. Best is trial 74 with value: 0.21802891883599546.
Fold 1 IBS: 0.23097652216923997
Fold 2 IBS: 0.19214396438491765
Fold 3 IBS: 0.26995098233323767
Fold 4 IBS: 0.21648924342245823
Fold 5 IBS: 0.21382651114907483
[I 2024-04-15 17:26:45,823] Trial 77 finished with value: 0.2246774446917857 and parameters: {'subsample': 0.1250123010159347, 'dropout_rate': 0.23331782495639364, 'n_estimators': 44, 'learning_rate': 0.07309219922663512}. Best is trial 74 with value: 0.21802891883599546.
Fold 1 IBS: 0.2426433538820121
Fold 2 IBS: 0.21530675860757498
Fold 3 IBS: 0.30748030302386065
Fold 4 IBS: 0.23702690393356035
Fold 5 IBS: 0.2505278119797517
[I 2024-04-15 17:26:46,647] Trial 78 finished with value: 0.25059702628535196 and parameters: {'subsample'

Fold 2 IBS: 0.19781108710453746
Fold 3 IBS: 0.26824641091261114
Fold 4 IBS: 0.2219303508164059
Fold 5 IBS: 0.21776871147453294
[I 2024-04-15 17:27:07,946] Trial 96 finished with value: 0.22550967088187218 and parameters: {'subsample': 0.25417895487176095, 'dropout_rate': 0.4365555547853381, 'n_estimators': 40, 'learning_rate': 0.06750679905117915}. Best is trial 74 with value: 0.21802891883599546.
Fold 1 IBS: 0.23012845095006443
Fold 2 IBS: 0.21146558867865006
Fold 3 IBS: 0.23279549174045716
Fold 4 IBS: 0.22266577695527393
Fold 5 IBS: 0.21241427656033215
[I 2024-04-15 17:27:08,329] Trial 97 finished with value: 0.22189391697695554 and parameters: {'subsample': 0.33846670341980367, 'dropout_rate': 0.31372183756474226, 'n_estimators': 11, 'learning_rate': 0.08189974172037857}. Best is trial 74 with value: 0.21802891883599546.
Fold 1 IBS: 0.22449452686444515
Fold 2 IBS: 0.19899271420825793
Fold 3 IBS: 0.2512552344527812
Fold 4 IBS: 0.21747374636329503
Fold 5 IBS: 0.20877722392455747
[I 20

In [65]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [66]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.664
train_ibs:  0.218


#### Test

In [67]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [68]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10222075562837932,
                                              learning_rate=0.07704376427520507,
                                              n_estimators=290,
                                              random_state=123,
                                              subsample=0.10832939724965542)

C-index score: 0.528


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3788374747846416,
                                              learning_rate=0.07386328482560535,
                                              n_estimators=22, random_state=123,
                                              subsample=0.10090556122023636)

IBS: 0.239


In [69]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [70]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.830,1.0
ExtraSurvivalTrees,0.764,2.0
GradientBoosting,0.713,3.0
CoxElastic,0.687,4.0
CoxPH,0.686,5.0
CoxLasso,0.685,6.0
ComponentwiseGradientBoosting,0.664,7.0
CoxRidge,0.659,8.0


In [71]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.202,1.0
ExtraSurvivalTrees,0.207,2.0
CoxLasso,0.211,3.5
CoxElastic,0.211,3.5
CoxPH,0.212,5.0
ComponentwiseGradientBoosting,0.218,6.0
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [72]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.544,1.0
ExtraSurvivalTrees,0.543,2.0
CoxRidge,0.534,3.0
GradientBoosting,0.532,4.0
ComponentwiseGradientBoosting,0.528,5.0
CoxPH,0.524,6.0
CoxLasso,0.523,7.5
CoxElastic,0.523,7.5


In [73]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxRidge,0.229,1.5
GradientBoosting,0.229,1.5
ComponentwiseGradientBoosting,0.239,3.0
Randomsurvivalforest,0.247,4.0
ExtraSurvivalTrees,0.256,5.0
CoxLasso,0.290,6.5
CoxElastic,0.290,6.5
CoxPH,0.292,8.0


In [74]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/yeojohnson/rent/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_yeojohnson_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [75]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-15
